# TFIDF

## Link git

In [3]:
# CELL 1 — Runtime information (GPU is optional for this cell)
import importlib.metadata as md, os, platform, shutil, sys
from pathlib import Path
print('Python:', sys.version)
print('OS:', platform.platform())
try:
    import torch
    print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except ImportError:
    print('torch: not installed')
for package in ('scikit-learn', 'transformers'):
    try: print(package + ':', md.version(package))
    except md.PackageNotFoundError: print(package + ': not installed')
print('Disk free GiB:', round(shutil.disk_usage('/content' if Path('/content').exists() else '.').free / 1024**3, 2))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
OS: Linux-6.6.159+-x86_64-with-glibc2.39
torch: 2.9.0+cu128 | CUDA available: True
GPU: NVIDIA L4
VRAM GiB: 22.03
scikit-learn: 1.7.2
transformers: 4.57.1
Disk free GiB: 58.71


In [4]:
# CELL 2 — GitHub configuration. Set a full commit SHA; no branch/latest fallback.
REPO_URL = 'https://github.com/khanhtrphuocbao/ABSA-lipstick-EC.git'
COMMIT_SHA = '4a361e5b5322c854ea0b860ee90f7a8ffc192e44'
WORKDIR = '/content/absa-project'
DATA_DIR = ''  # optional separate directory containing the four frozen CSV files
MODEL_ASSETS_DIR = ''  # optional local/Drive path to exact fingerprinted PhoBERT tokenizer and VnCoreNLP files
GOOGLE_DRIVE_ARTIFACT_DIR = ''  # optional; configured in CELL 7
MOUNT_GOOGLE_DRIVE = False
import re
if not REPO_URL.strip(): raise ValueError('Set REPO_URL before cloning')
if not re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA): raise ValueError('Set COMMIT_SHA to a full 40-character SHA')

In [5]:
import subprocess

clone_dir = Path(WORKDIR)
project_dir_name = (
    'Aspect-based-Sentiment-Analysis-for-Vietnamese-Reviews-about-Beauty-Product-on-E-commerce-Websites'
)

if clone_dir.exists():
    if not (clone_dir / '.git').exists():
        raise RuntimeError(f'WORKDIR exists but is not a git checkout: {clone_dir}')
else:
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(clone_dir)], check=True)

subprocess.run(['git', 'fetch', '--no-tags', 'origin', COMMIT_SHA],
               cwd=clone_dir, check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT_SHA],
               cwd=clone_dir, check=True)

head = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=clone_dir,
    check=True, capture_output=True, text=True
).stdout.strip()

if head.lower() != COMMIT_SHA.lower():
    raise RuntimeError(f'HEAD {head} does not match requested commit')

dirty = subprocess.run(
    ['git', 'status', '--porcelain'], cwd=clone_dir,
    check=True, capture_output=True, text=True
).stdout
if dirty.strip():
    raise RuntimeError('Checkout is dirty:\n' + dirty)

repo_dir = clone_dir / project_dir_name
if not (repo_dir / 'protocols').is_dir():
    raise FileNotFoundError(f'Project folder not found: {repo_dir}')

print('Verified clean checkout:', head)
print('Project directory:', repo_dir)

Verified clean checkout: 4a361e5b5322c854ea0b860ee90f7a8ffc192e44
Project directory: /content/absa-project/Aspect-based-Sentiment-Analysis-for-Vietnamese-Reviews-about-Beauty-Product-on-E-commerce-Websites


## Run this then restart

In [9]:
lock = repo_dir / "protocols/experiments/modeling_requirements_v1.lock"

subprocess.run(
    [
        sys.executable, "-u", "-m", "pip", "install", "-v",
        "--require-hashes", "-r", str(lock),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'pip', 'install', '-v', '--require-hashes', '-r', '/content/absa-project/Aspect-based-Sentiment-Analysis-for-Vietnamese-Reviews-about-Beauty-Product-on-E-commerce-Websites/protocols/experiments/modeling_requirements_v1.lock'], returncode=0)

In [4]:
# CELL 4 — Dataset location and byte hashes. This cell never cleans or rewrites data.
import hashlib, json
repo_data = repo_dir / 'data'
data_dir = Path(DATA_DIR).expanduser() if DATA_DIR else repo_data
if DATA_DIR and not data_dir.exists() and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
protocol = json.loads((repo_dir / 'protocols/evaluation_protocol_v1.yaml').read_text(encoding='utf-8'))
for split, meta in protocol['source_files'].items():
    source = data_dir / Path(meta['path']).name
    if not source.is_file(): raise FileNotFoundError(f'Missing frozen {split} CSV: {source}. If staged on Drive, mount in CELL 7, set DATA_DIR, then rerun this cell.')
    actual = hashlib.sha256(source.read_bytes()).hexdigest()
    if actual != meta['sha256']: raise ValueError(f'Frozen CSV SHA-256 mismatch: {source}')
    print(split, meta['rows'], actual)
print('Dataset bytes verified; use only official train/validation for fitting and selection.')

full 16227 3bda8deecc84d94fff526ad092cadc1dc4a4487e6d1a8c54871e53da95ec780e
train 12981 19449cceea317269304464ef7c6c72d5dc70548c6006686d1506e1880bfb9976
val 1623 7bbc93fad8db9c78a83f75800119cad65bf65b8f0241c36228d7cbe13d1535b3
test 1623 90d1bf0b64105c89346f60d974c5c72c0208c70458c40d5b089ac48486d43d08
Dataset bytes verified; use only official train/validation for fitting and selection.


In [5]:
# CELL 5 — Validate registration v2 and the exact modeling runtime; no packages are installed here.
import hashlib, importlib.metadata as md, json, platform, unicodedata
registration_path = repo_dir / 'protocols/experiments/experiment_registration_v2.json'
lock_path = repo_dir / 'protocols/experiments/experiment_registration_lock_v2.json'
registration = json.loads(registration_path.read_text(encoding='utf-8'))
registration_lock = json.loads(lock_path.read_text(encoding='utf-8'))
if registration_lock['registration_id'] != registration['registration_id']:
    raise RuntimeError('Registration v2 ID does not match its integrity lock')
for relative, expected in registration_lock['artifacts'].items():
    target = repo_dir / relative
    actual = hashlib.sha256(target.read_bytes()).hexdigest() if target.is_file() else None
    if actual != expected: raise RuntimeError(f'Registration v2 artifact hash mismatch: {relative}')
expected_runtime = registration['runtime']['modeling']
problems = []
if platform.python_version() != expected_runtime['python']:
    problems.append(f"Python {platform.python_version()} != registered {expected_runtime['python']}")
if unicodedata.unidata_version != expected_runtime['unicode']:
    problems.append(f"Unicode {unicodedata.unidata_version} != registered {expected_runtime['unicode']}")
for package, version in expected_runtime['package_versions'].items():
    try: actual = md.version(package)
    except md.PackageNotFoundError: actual = 'missing'
    print(package, 'installed=', actual, 'registered=', version)
    if actual != version: problems.append(f'{package} {actual} != {version}')
lock = repo_dir / expected_runtime['resolved_environment']
tokenizer_pins = repo_dir / expected_runtime['installed_tokenizer_environment']
print('Hash-locked package install command (not executed): python -m pip install --require-hashes -r', lock)
print('Tokenizer pins (already included in the shared lock):', tokenizer_pins)
if problems: raise RuntimeError('Registered modeling environment is incompatible: ' + '; '.join(problems))
print('PASS: registration v2 and modeling runtime verified. Data/evaluator verification remains v1.')


scikit-learn installed= 1.7.2 registered= 1.7.2
torch installed= 2.9.0 registered= 2.9.0
transformers installed= 4.57.1 registered= 4.57.1
Hash-locked package install command (not executed): python -m pip install --require-hashes -r /content/absa-project/Aspect-based-Sentiment-Analysis-for-Vietnamese-Reviews-about-Beauty-Product-on-E-commerce-Websites/protocols/experiments/modeling_requirements_v1.lock
Tokenizer pins (already included in the shared lock): /content/absa-project/Aspect-based-Sentiment-Analysis-for-Vietnamese-Reviews-about-Beauty-Product-on-E-commerce-Websites/reports/modeling_preflight/tokenizer_environment_v1.txt
PASS: registration v2 and modeling runtime verified. Data/evaluator verification remains v1.


In [6]:
# CELL 6 — Portable byte integrity only; does not rerun exact-runtime normalization.
portable_cmd = [sys.executable, str(repo_dir / 'src/portable_verification.py'),
                '--root', str(repo_dir), '--commit-sha', COMMIT_SHA]
if MODEL_ASSETS_DIR: portable_cmd += ['--assets-dir', str(Path(MODEL_ASSETS_DIR).expanduser())]
portable = subprocess.run(portable_cmd,
                          cwd=repo_dir, check=True, capture_output=True, text=True)
portable_result = json.loads(portable.stdout)
assert portable_result['exact_runtime_reproduced'] is False
print(json.dumps(portable_result, indent=2))
print('Historical local exact-runtime receipt:', portable_result['gate_runtime']['historical_receipt'])

{
  "asset_files_checked": [],
  "asset_revisions": {
    "phobert": "01daacda68afe13d83023d16ec647239e344a1e6",
    "vncorenlp": "62bbc58fe5d113c898eae112656be97dcf50b3a0"
  },
  "exact_runtime_reproduced": false,
  "freeze_lock_sha256": "16e9b76b6a71530d08704cadc2d1ddeddf54ca57615c35f8e650ccc57ac86ee8",
  "gate_runtime": {
    "historical_receipt": "reports/modeling_preflight/reproducibility_gate_v1.json",
    "python": "3.15.1",
    "unicode": "17.0.0"
  },
  "git_head": "4a361e5b5322c854ea0b860ee90f7a8ffc192e44",
  "modeling_runtime": {
    "python": "3.13.9",
    "unicode": "15.1.0"
  },
  "note": "Portable byte hashes only; this does not reproduce Python 3.15.1 / Unicode 17 normalization.",
  "portable_immutable_hashes": "PASS",
  "registration_lock_sha256": "7279166cb41c3676e835e27bfd3bc53e9330889145fc73a4e91b96fea16a83e4",
  "source_csv_sha256": {
    "full": "3bda8deecc84d94fff526ad092cadc1dc4a4487e6d1a8c54871e53da95ec780e",
    "test": "90d1bf0b64105c89346f60d974c5c72c0208c70

## Mount drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# CELL 7 — Optional Google Drive mount for durable artifacts or separately staged private data.
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('Google Drive mounting is available only in Colab') from exc
    if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
artifact_dir = Path(GOOGLE_DRIVE_ARTIFACT_DIR).expanduser() if GOOGLE_DRIVE_ARTIFACT_DIR else None
if artifact_dir:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    print('Durable artifact backup directory:', artifact_dir)
    print('Use /content local disk for active data and temporary checkpoints; copy finalized artifacts to Drive explicitly.')
else:
    print('Drive not mounted. Set MOUNT_GOOGLE_DRIVE=True and GOOGLE_DRIVE_ARTIFACT_DIR to opt in.')

Drive not mounted. Set MOUNT_GOOGLE_DRIVE=True and GOOGLE_DRIVE_ARTIFACT_DIR to opt in.


In [10]:
from pathlib import Path

repo_dir = Path(
    "/content/absa-project/Aspect-based-Sentiment-Analysis-for-Vietnamese-Reviews-about-Beauty-Product-on-E-commerce-Websites"
)
local_runs = Path("/content/absa-runs")
data_dir = Path("/content/drive/MyDrive/absa-data")

local_runs.mkdir(parents=True, exist_ok=True)

assert repo_dir.is_dir(), f"Không tìm thấy repo: {repo_dir}"
assert data_dir.is_dir(), f"Không tìm thấy thư mục dữ liệu: {data_dir}"

print("repo_dir:", repo_dir)
print("local_runs:", local_runs)
print("data_dir:", data_dir)

repo_dir: /content/absa-project/Aspect-based-Sentiment-Analysis-for-Vietnamese-Reviews-about-Beauty-Product-on-E-commerce-Websites
local_runs: /content/absa-runs
data_dir: /content/drive/MyDrive/absa-data


## Training

In [ ]:
# CELL 8 — CPU synthetic smoke only; generated artificial rows, no official CSV/model weights.
smoke = subprocess.run([sys.executable, '-m', 'src.training.synthetic_smoke', '--cpu-only'],
                       cwd=repo_dir, check=True, capture_output=True, text=True)
print(smoke.stdout)
if smoke.stderr: print(smoke.stderr)

In [ ]:
# CELL 9 — Future launch gate status. This cell only displays registration; it never trains.
for family, config in registration['families'].items():
    print(family, [candidate['id'] for candidate in config['candidates']])
print('Seeds:', registration['seeds'], '| maximum registered fits:', registration['budget']['maximum_main_training_runs'])
print('Neural target: single CUDA GPU >=16 GiB; verify driver/CUDA and deterministic behavior at launch.')
print('TRAIN-04 registered runner available:', (repo_dir / 'src/training/registered_run.py').is_file())
print('This notebook never starts a fit. The explicit TRAIN-04 CLI requires a clean authorized manifest and a separate process.')
print('Pending: TRAIN-02B GPU smoke; neural run requires pinned assets and a CUDA GPU with at least 16 GiB VRAM.')

In [13]:
import sklearn, torch, transformers

print("scikit-learn:", sklearn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

scikit-learn: 1.7.2
torch: 2.9.0+cu128
transformers: 4.57.1


In [18]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

run_ids = [
    "TFIDF-01-seed-42",
    "TFIDF-01-seed-2026",
    "TFIDF-01-seed-3407",
    "TFIDF-02-seed-42",
    "TFIDF-02-seed-2026",
    "TFIDF-02-seed-3407",
]

local_runs = Path(local_runs)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# Bước 1: tạo manifest cho từng run nếu chưa có.
for run_id in run_ids:
    candidate, seed_text = run_id.rsplit("-seed-", 1)
    seed = int(seed_text)
    run_dir = local_runs / run_id
    manifest = run_dir / "run_manifest.json"

    if not manifest.exists():
        print(f"Prepare {run_id}", flush=True)
        subprocess.run([
            sys.executable, "-u", "-m", "src.training.run_manifest", "prepare",
            "--candidate", candidate,
            "--seed", str(seed),
            "--run-id", run_id,
            "--artifact-root", str(local_runs),
            "--data-dir", str(data_dir),
        ], cwd=repo_dir, check=True)

    manifest_data = json.loads(manifest.read_text())
    if (manifest_data.get("run_id") != run_id
            or manifest_data.get("candidate_id") != candidate
            or manifest_data.get("seed") != seed):
        raise RuntimeError(f"Manifest không khớp candidate/seed/run ID: {manifest}")

# Bước 2: authorize từng manifest nếu chưa authorize.
for run_id in run_ids:
    manifest = local_runs / run_id / "run_manifest.json"
    authorization = local_runs / run_id / "run_authorization.json"

    if authorization.exists():
        auth_data = json.loads(authorization.read_text())
        if (auth_data.get("run_id") != run_id
                or auth_data.get("manifest_sha256") != sha256(manifest)):
            raise RuntimeError(f"Authorization cũ không khớp manifest: {run_id}")
        print(f"Đã authorize: {run_id}")
        continue

    print(f"Authorize {run_id}", flush=True)
    subprocess.run([
        sys.executable, "-u", "-m", "src.training.run_manifest", "authorize",
        "--manifest", str(manifest),
        "--confirm-run-id", run_id,
        "--data-dir", str(data_dir),
    ], cwd=repo_dir, check=True)

print("Đã prepare và authorize đủ 5 run. Chưa có model nào được fit.")

Prepare TFIDF-01-seed-42
Authorize TFIDF-01-seed-42
Đã authorize: TFIDF-01-seed-2026
Đã authorize: TFIDF-01-seed-3407
Đã authorize: TFIDF-02-seed-42
Đã authorize: TFIDF-02-seed-2026
Đã authorize: TFIDF-02-seed-3407
Đã prepare và authorize đủ 5 run. Chưa có model nào được fit.


In [20]:
import hashlib
import json
import subprocess
import sys
import os
from pathlib import Path

run_ids = [
    "TFIDF-01-seed-42",
    "TFIDF-01-seed-2026",
    "TFIDF-01-seed-3407",
    "TFIDF-02-seed-42",
    "TFIDF-02-seed-2026",
    "TFIDF-02-seed-3407",
]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Chạy từng run; không tự chạy lại nếu đã có kết quả.
for run_id in run_ids:
    run_dir = Path(local_runs) / run_id
    manifest = run_dir / "run_manifest.json"
    authorization = run_dir / "run_authorization.json"
    result_path = run_dir / "run_result.json"

    if not manifest.is_file() or not authorization.is_file():
        raise RuntimeError(
            f"{run_id}: thiếu run_manifest.json hoặc run_authorization.json. "
            "Hãy prepare và authorize run này trước."
        )

    if result_path.exists():
        raise RuntimeError(
            f"{run_id}: đã có run_result.json. Run là write-once; "
            "không chạy lại cùng run ID."
        )

    print(f"\n===== RUN {run_id} =====", flush=True)


    seed = int(run_id.rsplit("-seed-", 1)[1])

    run_env = os.environ.copy()
    run_env["PYTHONHASHSEED"] = str(seed)
    run_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    proc = subprocess.run([
        sys.executable, "-u", "-m", "src.training.registered_run", "run",
        "--manifest", str(manifest),
        "--run-id", run_id,
        "--data-dir", str(data_dir),
    ], cwd=repo_dir, env=run_env, check=False)

    # Không dựa riêng vào exit code: xác nhận các artifact hoàn tất.
    if not result_path.is_file():
        raise RuntimeError(
            f"{run_id}: không có run_result.json; tiến trình trả exit code {proc.returncode}."
        )

    result = json.loads(result_path.read_text())
    checkpoint = Path(result["checkpoint_path"])
    predictions = Path(result["validation_predictions_path"])
    completed_events = [
        p for p in run_dir.glob("run-event-*-completed.json")
    ]

    if result.get("status") != "COMPLETED" or result.get("test_evaluated") is not False:
        raise RuntimeError(f"{run_id}: trạng thái kết quả không hợp lệ.")

    if not checkpoint.is_file() or sha256(checkpoint) != result["checkpoint_sha256"]:
        raise RuntimeError(f"{run_id}: checkpoint thiếu hoặc hash không khớp.")

    if (not predictions.is_file()
            or sha256(predictions) != result["validation_predictions_sha256"]):
        raise RuntimeError(f"{run_id}: validation predictions thiếu hoặc hash không khớp.")

    matching_event = False
    result_hash = sha256(result_path)
    for event_path in completed_events:
        event = json.loads(event_path.read_text())
        if event.get("run_result_sha256") == result_hash:
            matching_event = True
            break

    if not matching_event:
        raise RuntimeError(f"{run_id}: không tìm thấy completed event khớp hash kết quả.")

    print(
        f"VERIFIED {run_id}: "
        f"Macro-F1={result['val_joint_macro_f1']:.6f}, "
        f"Micro-F1={result['val_joint_micro_f1']:.6f}, "
        f"exit_code={proc.returncode}",
        flush=True,
    )

# Aggregate chỉ sau khi 5 run trên đã xác minh; runner cũng kiểm tra đủ 6 run
# (bao gồm TFIDF-01-seed-42 đã chạy trước đó).
selection_path = Path(local_runs) / "family_selection" / "tfidf_logistic.json"
if selection_path.exists():
    raise RuntimeError(
        f"Aggregate output đã tồn tại: {selection_path}. "
        "Không ghi đè kết quả write-once; hãy kiểm tra file hiện có."
    )

print("\n===== AGGREGATE TF-IDF =====", flush=True)
subprocess.run([
    sys.executable, "-u", "-m", "src.training.registered_run", "aggregate",
    "--family", "tfidf_logistic",
    "--artifact-root", str(local_runs),
], cwd=repo_dir, check=True)

print(f"\nAggregate hoàn tất: {selection_path}")


===== RUN TFIDF-01-seed-42 =====
VERIFIED TFIDF-01-seed-42: Macro-F1=0.487509, Micro-F1=0.844496, exit_code=0

===== RUN TFIDF-01-seed-2026 =====
VERIFIED TFIDF-01-seed-2026: Macro-F1=0.487509, Micro-F1=0.844496, exit_code=0

===== RUN TFIDF-01-seed-3407 =====
VERIFIED TFIDF-01-seed-3407: Macro-F1=0.487509, Micro-F1=0.844496, exit_code=0

===== RUN TFIDF-02-seed-42 =====
VERIFIED TFIDF-02-seed-42: Macro-F1=0.521098, Micro-F1=0.864883, exit_code=0

===== RUN TFIDF-02-seed-2026 =====
VERIFIED TFIDF-02-seed-2026: Macro-F1=0.521098, Micro-F1=0.864883, exit_code=0

===== RUN TFIDF-02-seed-3407 =====
VERIFIED TFIDF-02-seed-3407: Macro-F1=0.521098, Micro-F1=0.864883, exit_code=0

===== AGGREGATE TF-IDF =====

Aggregate hoàn tất: /content/absa-runs/family_selection/tfidf_logistic.json


In [24]:
import shutil
from pathlib import Path

# Định nghĩa đường dẫn nguồn và đích trên Google Drive
source_dir = Path("/content/absa-runs")
destination_dir = Path("/content/drive/MyDrive/absa-runs")

# Tiến hành copy thư mục
if source_dir.exists():
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
    print(f"Đã sao chép thành công toàn bộ thư mục từ {source_dir} sang {destination_dir}")
else:
    print(f"Thư mục nguồn không tồn tại: {source_dir}")

Đã sao chép thành công toàn bộ thư mục từ /content/absa-runs sang /content/drive/MyDrive/absa-runs


# BIGRU

In [20]:
import subprocess
import sys

# Gỡ torchvision mặc định của Colab đang xung đột với torch 2.9.0.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    check=True,
)

# Kiểm tra trong process mới; không tải weights và không train.
probe = subprocess.run(
    [
        sys.executable, "-u", "-c",
        "import torch, transformers; "
        "from transformers import RobertaModel; "
        "print('torch:', torch.__version__); "
        "print('transformers:', transformers.__version__); "
        "print('RobertaModel import: PASS')",
    ],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=False,
)

print("Exit code:", probe.returncode)
print("STDOUT:\n", probe.stdout or "(trống)")
print("STDERR:\n", probe.stderr or "(trống)")

Exit code: 0
STDOUT:
 torch: 2.9.0+cu128
transformers: 4.57.1
RobertaModel import: PASS

STDERR:
I0000 00:00:1793261349.286531    6844 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793261349.355787    6844 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793261351.852216    6844 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.



## Run this then restart

In [1]:
import hashlib
import os
import subprocess
import tarfile
import urllib.request
from pathlib import Path

url = (
    "https://github.com/adoptium/temurin17-binaries/releases/download/"
    "jdk-17.0.21%2B9/OpenJDK17U-jdk_x64_linux_hotspot_17.0.21_9.tar.gz"
)
expected_sha256 = "8b9b611d917e3dfd8419d3566b5949226f69b052fed7e109c1368add1dfe8711"

archive = Path("/content/OpenJDK17U-jdk_x64_linux_hotspot_17.0.21_9.tar.gz")
extract_root = Path("/content/temurin-17.0.21+9")
extract_root.mkdir(parents=True, exist_ok=True)

if not archive.exists():
    print("Đang tải JDK đã pin…")
    urllib.request.urlretrieve(url, archive)

actual_sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()
if actual_sha256 != expected_sha256:
    raise RuntimeError(f"JDK SHA-256 không khớp: {actual_sha256}")

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(extract_root, filter="data")

matches = list(extract_root.glob("jdk-17.0.21+9"))
if len(matches) != 1:
    raise RuntimeError(f"Không tìm thấy thư mục JDK mong đợi: {matches}")

jdk_home = matches[0]
os.environ["JAVA_HOME"] = str(jdk_home)
os.environ["PATH"] = str(jdk_home / "bin") + os.pathsep + os.environ["PATH"]

subprocess.run([str(jdk_home / "bin/java"), "-version"], check=True)
print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = /content/temurin-17.0.21+9/jdk-17.0.21+9


In [11]:
import json
import subprocess
import sys
from pathlib import Path

repo_dir = Path(repo_dir)
local_runs = Path(local_runs)
data_dir = Path(data_dir)
local_runs.mkdir(parents=True, exist_ok=True)

# Có thể đổi thành thư mục assets bạn đã chuẩn bị trên Drive.
assets_dir = (
    Path(MODEL_ASSETS_DIR).expanduser()
    if globals().get("MODEL_ASSETS_DIR")
    else Path("/content/absa-assets")
)
assets_dir.mkdir(parents=True, exist_ok=True)

assert repo_dir.is_dir(), f"Không tìm thấy repo: {repo_dir}"
assert data_dir.is_dir(), f"Không tìm thấy data_dir: {data_dir}"

import torch
assert torch.cuda.is_available(), "BiGRU/PhoBERT cần CUDA GPU."
gpu = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(gpu.total_memory / 1024**3, 2))
assert gpu.total_memory >= 16 * 1024**3, "Cần GPU có ít nhất 16 GiB VRAM."

# Chỉ tải/khôi phục các file tokenizer và VnCoreNLP có checksum đã đăng ký.
# Lệnh này KHÔNG tải PhoBERT pretrained weights.
subprocess.run(
    [
        sys.executable, "-u", "-m", "src.training.restore_runtime_assets",
        "--assets", str(assets_dir),
    ],
    cwd=repo_dir,
    check=True,
)

java = subprocess.run(
    ["java", "-version"], capture_output=True, text=True, check=False
)
print("Java version:\n", java.stdout, java.stderr)

GPU: NVIDIA L4
VRAM GiB: 22.03
Java version:
  openjdk version "17.0.21" 2026-10-20
OpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)
OpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)



In [12]:
import sys
from pathlib import Path

# Thêm repo_dir vào sys.path để Python tìm thấy package 'src'
if 'repo_dir' in globals():
    repo_path = str(Path(repo_dir).resolve())
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

from src.training.assets import verify_assets

asset_report = verify_assets(
    assets_dir, repo_dir, require_java_archive=False
)
print(asset_report)

{'phobert_revision': '01daacda68afe13d83023d16ec647239e344a1e6', 'vncorenlp_revision': '62bbc58fe5d113c898eae112656be97dcf50b3a0', 'checked_files': ['phobert/config.json', 'phobert/vocab.txt', 'phobert/bpe.codes', 'vncorenlp/VnCoreNLP-1.2.jar', 'vncorenlp/models/wordsegmenter/vi-vocab', 'vncorenlp/models/wordsegmenter/wordsegmenter.rdr'], 'java_release': 'jdk-17.0.21+9', 'java_executable': '/content/temurin-17.0.21+9/jdk-17.0.21+9/bin/java', 'java_sha256': 'a2195ae9f8a91b82f53df7e1d1b9a650937ebbabbf67d0240c6378b4fbd9990b', 'java_version_output': 'openjdk version "17.0.21" 2026-10-20\nOpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)\nOpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)'}


In [24]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    check=True,
)

p = subprocess.run(
    [
        sys.executable, "-u", "-c",
        "import torch, transformers; "
        "from transformers import RobertaModel; "
        "print('torch:', torch.__version__); "
        "print('transformers:', transformers.__version__); "
        "print('RobertaModel import: PASS')",
    ],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=False,
)

print("Exit code:", p.returncode)
print("STDOUT:\n", p.stdout or "(trống)")
print("STDERR:\n", p.stderr or "(trống)")

Exit code: 0
STDOUT:
 torch: 2.9.0+cu128
transformers: 4.57.1
RobertaModel import: PASS

STDERR:
I0000 00:00:1793247080.384681   14473 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793247080.459506   14473 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793247083.093850   14473 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.



##Traning

In [23]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

families = {
    "bigru_conv1d": ["BIGRU-01", "BIGRU-02"],
    "phobert": ["PHOBERT-01", "PHOBERT-02"],
}
seeds = [42, 2026, 3407]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def cli(args):
    p = subprocess.run(
        [sys.executable, "-u", *args],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        check=False,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0:
        raise RuntimeError(f"CLI thất bại, exit code={p.returncode}: {args}")
    return p

run_ids = []
for candidates in families.values():
    for candidate in candidates:
        for seed in seeds:
            run_id = f"{candidate}-seed-{seed}"
            run_ids.append(run_id)
            run_dir = local_runs / run_id
            manifest = run_dir / "run_manifest.json"
            authorization = run_dir / "run_authorization.json"

            if not manifest.exists():
                if run_dir.exists() and any(run_dir.iterdir()):
                    raise RuntimeError(
                        f"{run_id}: thư mục đã có file nhưng thiếu manifest; "
                        "hãy kiểm tra artifact, không ghi đè."
                    )
                cli([
                    "-m", "src.training.run_manifest", "prepare",
                    "--candidate", candidate,
                    "--seed", str(seed),
                    "--run-id", run_id,
                    "--artifact-root", str(local_runs),
                    "--data-dir", str(data_dir),
                ])

            m = json.loads(manifest.read_text())
            if (m.get("run_id") != run_id
                    or m.get("candidate_id") != candidate
                    or m.get("seed") != seed):
                raise RuntimeError(f"{run_id}: manifest không khớp candidate/seed.")

            common = [
                "--manifest", str(manifest),
                "--assets-dir", str(assets_dir),
                "--data-dir", str(data_dir),
            ]

            # Validate trước; chỉ authorize nếu gate PASS.
            cli(["-m", "src.training.run_manifest", "validate", *common])

            if authorization.exists():
                a = json.loads(authorization.read_text())
                if (a.get("run_id") != run_id
                        or a.get("manifest_sha256") != sha256(manifest)):
                    raise RuntimeError(f"{run_id}: authorization không khớp manifest.")
                print("Đã authorize:", run_id)
            else:
                cli([
                    "-m", "src.training.run_manifest", "authorize",
                    *common,
                    "--confirm-run-id", run_id,
                ])

print(f"Đã kiểm tra {len(run_ids)} manifest. Chưa bắt đầu training.")

{
  "report": "/content/absa-runs/BIGRU-01-seed-42/validation-20261029T083541686977Z.json",
  "status": "PASS",
  "run_id": "BIGRU-01-seed-42",
  "candidate_id": "BIGRU-01",
  "seed": 42,
  "blockers": [],
  "fit_started": false,
  "validated_utc": "2026-10-29T08:35:41.686957+00:00",
  "observations": {
    "git": {
      "commit": "4a361e5b5322c854ea0b860ee90f7a8ffc192e44",
      "dirty": false,
      "dirty_sha256": null
    },
    "runtime": {
      "python": "3.13.16",
      "unicode": "15.1.0",
      "packages": {
        "scikit-learn": "1.7.2",
        "torch": "2.9.0",
        "transformers": "4.57.1"
      }
    },
    "artifacts": {
      "protocols/experiments/experiment_registration_lock_v1.json": {
        "expected_sha256": "7279166cb41c3676e835e27bfd3bc53e9330889145fc73a4e91b96fea16a83e4",
        "observed_sha256": "7279166cb41c3676e835e27bfd3bc53e9330889145fc73a4e91b96fea16a83e4"
      },
      "protocols/experiments/experiment_registration_v1.json": {
        "expecte

RuntimeError: CLI thất bại, exit code=2: ['-m', 'src.training.run_manifest', 'validate', '--manifest', '/content/absa-runs/PHOBERT-01-seed-42/run_manifest.json', '--assets-dir', '/content/absa-assets', '--data-dir', '/content/drive/MyDrive/absa-data']

In [15]:
import json
from pathlib import Path

run_dir = Path(local_runs) / "PHOBERT-01-seed-42"

for p in sorted(run_dir.rglob("*")):
    print(("DIR " if p.is_dir() else "FILE"), p.relative_to(run_dir))

for p in sorted(run_dir.glob("run-event-*.json")):
    print(f"\n--- {p.name} ---")
    print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False))

DIR  checkpoints
FILE run-event-20261029T080334734723Z-running.json
FILE run-event-20261029T080350464503Z-failed.json
FILE run_authorization.json
FILE run_manifest.json
FILE validation-20261029T080258936100Z.json
FILE validation-20261029T080618807057Z.json
FILE validation-20261029T081216610235Z.json
FILE validation-20261029T081406787718Z.json

--- run-event-20261029T080334734723Z-running.json ---
{
  "candidate_id": "PHOBERT-01",
  "run_id": "PHOBERT-01-seed-42",
  "seed": 42,
  "source_split_policy": {
    "fit": "train",
    "selection": "val",
    "test_loaded": false
  },
  "state": "RUNNING",
  "timestamp_utc": "2026-10-29T08:03:34.734723+00:00"
}

--- run-event-20261029T080350464503Z-failed.json ---
{
  "error": "Could not import module 'RobertaModel'. Are this object's requirements defined correctly?",
  "error_type": "ModuleNotFoundError",
  "state": "FAILED",
  "timestamp_utc": "2026-10-29T08:03:50.464503+00:00"
}


In [17]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

run_ids = [
    "PHOBERT-01-seed-42",
]
for run_id in run_ids:
    run_dir = local_runs / run_id
    manifest = run_dir / "run_manifest.json"
    authorization = run_dir / "run_authorization.json"
    result_path = run_dir / "run_result.json"

    if not manifest.is_file() or not authorization.is_file():
        raise RuntimeError(f"{run_id}: thiếu manifest hoặc authorization.")

    # Run là write-once: không chạy lại ID đã có kết quả hoặc event.
    if result_path.exists() or list(run_dir.glob("run-event-*.json")):
        raise RuntimeError(
            f"{run_id}: đã có kết quả/event. Kiểm tra artifact trước, không chạy lại."
        )

    seed = int(run_id.rsplit("-seed-", 1)[1])
    run_env = os.environ.copy()
    run_env["PYTHONHASHSEED"] = str(seed)
    run_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    print(f"\n===== RUN {run_id} =====", flush=True)
    proc = subprocess.run(
        [
            sys.executable, "-u", "-m", "src.training.registered_run", "run",
            "--manifest", str(manifest),
            "--run-id", run_id,
            "--assets-dir", str(assets_dir),
            "--data-dir", str(data_dir),
        ],
        cwd=repo_dir,
        env=run_env,
        check=False,
    )

    if not result_path.is_file():
        raise RuntimeError(
            f"{run_id}: không có run_result.json; exit code={proc.returncode}. "
            "Kiểm tra run events trước khi quyết định bước tiếp."
        )

    result = json.loads(result_path.read_text())
    checkpoint = Path(result["checkpoint_path"])
    predictions = Path(result["validation_predictions_path"])

    if result.get("status") != "COMPLETED" or result.get("test_evaluated") is not False:
        raise RuntimeError(f"{run_id}: trạng thái kết quả không hợp lệ.")
    if not checkpoint.is_file() or sha256(checkpoint) != result["checkpoint_sha256"]:
        raise RuntimeError(f"{run_id}: checkpoint thiếu hoặc hash sai.")
    if (not predictions.is_file()
            or sha256(predictions) != result["validation_predictions_sha256"]):
        raise RuntimeError(f"{run_id}: validation predictions thiếu hoặc hash sai.")

    result_hash = sha256(result_path)
    completed = False
    for event_path in run_dir.glob("run-event-*-completed.json"):
        event = json.loads(event_path.read_text())
        if event.get("run_result_sha256") == result_hash:
            completed = True
            break
    if not completed:
        raise RuntimeError(f"{run_id}: completed event không khớp hash kết quả.")

    print(
        f"VERIFIED {run_id}: "
        f"Macro-F1={result['val_joint_macro_f1']:.6f}, "
        f"Micro-F1={result['val_joint_micro_f1']:.6f}, "
        f"exit_code={proc.returncode}",
        flush=True,
    )


===== RUN PHOBERT-01-seed-42 =====
VERIFIED PHOBERT-01-seed-42: Macro-F1=0.522911, Micro-F1=0.863607, exit_code=0


##Trace lỗi

In [18]:
import json
import os
import subprocess
import sys
from pathlib import Path

run_dir = Path(local_runs) / "PHOBERT-01-seed-42"

print("=== RUN EVENTS ===")
events = sorted(run_dir.glob("run-event-*.json"))
if not events:
    print("Không tìm thấy event.")
for path in events:
    print(f"\n--- {path.name} ---")
    print(json.dumps(json.loads(path.read_text()), indent=2, ensure_ascii=False))

print("\n=== ROBERTA IMPORT PROBE ===")
probe_env = os.environ.copy()
probe_env["PYTHONHASHSEED"] = "42"
probe_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

probe = """
import importlib.metadata as md
for name in ("torch", "torchvision", "transformers"):
    try:
        print(name, md.version(name))
    except md.PackageNotFoundError:
        print(name, "not installed")
from transformers import RobertaModel
print("RobertaModel import: PASS")
"""

p = subprocess.run(
    [sys.executable, "-u", "-c", probe],
    cwd=repo_dir,
    env=probe_env,
    capture_output=True,
    text=True,
    check=False,
)
print("Exit code:", p.returncode)
print("STDOUT:\n", p.stdout or "(trống)")
print("STDERR:\n", p.stderr or "(trống)")


=== RUN EVENTS ===

--- run-event-20261029T082142686784Z-running.json ---
{
  "candidate_id": "PHOBERT-01",
  "run_id": "PHOBERT-01-seed-42",
  "seed": 42,
  "source_split_policy": {
    "fit": "train",
    "selection": "val",
    "test_loaded": false
  },
  "state": "RUNNING",
  "timestamp_utc": "2026-10-29T08:21:42.686784+00:00"
}

--- run-event-20261029T083127465103Z-completed.json ---
{
  "run_result_sha256": "0c596800c4e53f0054eab153406e109be421d764c3e148ec1c7c371272454b53",
  "state": "COMPLETED",
  "timestamp_utc": "2026-10-29T08:31:27.465103+00:00"
}

=== ROBERTA IMPORT PROBE ===
Exit code: 0
STDOUT:
 torch 2.9.0
torchvision not installed
transformers 4.57.1
RobertaModel import: PASS

STDERR:
I0000 00:00:1793262771.099302   14293 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:

In [38]:
import subprocess
import sys

# Gỡ torchvision; không thay đổi torch hoặc transformers.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    check=True,
)

# Kiểm tra bằng process mới, không tải weights và không train.
probe = """
import importlib.metadata as md
import torch
import transformers
from transformers import RobertaModel

print("torch metadata:", md.version("torch"))
print("torch runtime:", torch.__version__)
print("transformers:", transformers.__version__)
print("RobertaModel import: PASS")
"""

p = subprocess.run(
    [sys.executable, "-u", "-c", probe],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=False,
)
print("Exit code:", p.returncode)
print("STDOUT:\\n", p.stdout or "(trống)")
print("STDERR:\\n", p.stderr or "(trống)")

Exit code: 0
STDOUT:\n torch metadata: 2.9.0
torch runtime: 2.9.0+cu128
transformers: 4.57.1
RobertaModel import: PASS

STDERR:\n WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1793160494.612687   26148 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793160494.685145   26148 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793160497.069949   26148 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different 

In [20]:
import subprocess
import sys
from pathlib import Path

for family, candidates in families.items():
    expected = [
        f"{candidate}-seed-{seed}"
        for candidate in candidates
        for seed in seeds
    ]
    missing = [
        run_id for run_id in expected
        if not (local_runs / run_id / "run_result.json").is_file()
    ]
    if missing:
        print(f"{family}: chưa aggregate; thiếu {missing}")
        continue

    output_path = local_runs / "family_selection" / f"{family}.json"
    if output_path.exists():
        print(f"{family}: aggregate output đã có, không ghi đè: {output_path}")
        continue

    p = subprocess.run(
        [
            sys.executable, "-u", "-m", "src.training.registered_run", "aggregate",
            "--family", family,
            "--artifact-root", str(local_runs),
        ],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        check=False,
    )
    print("STDOUT:\n", p.stdout or "(trống)")
    print("STDERR:\n", p.stderr or "(trống)")
    if p.returncode != 0:
        raise RuntimeError(f"Aggregate {family} lỗi, exit code={p.returncode}")

print("Aggregate output nằm trong:", local_runs / "family_selection")

bigru_conv1d: chưa aggregate; thiếu ['BIGRU-01-seed-42', 'BIGRU-01-seed-2026', 'BIGRU-01-seed-3407', 'BIGRU-02-seed-42', 'BIGRU-02-seed-2026', 'BIGRU-02-seed-3407']
phobert: chưa aggregate; thiếu ['PHOBERT-01-seed-2026', 'PHOBERT-01-seed-3407', 'PHOBERT-02-seed-42', 'PHOBERT-02-seed-2026', 'PHOBERT-02-seed-3407']
Aggregate output nằm trong: /content/absa-runs/family_selection


##Save run results

In [21]:
import shutil
from pathlib import Path

# Định nghĩa đường dẫn nguồn và đích trên Google Drive
source_dir = Path("/content/absa-runs")
destination_dir = Path("/content/drive/MyDrive/absa-runs")

# Tiến hành copy thư mục
if source_dir.exists():
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
    print(f"Đã sao chép thành công toàn bộ thư mục từ {source_dir} sang {destination_dir}")
else:
    print(f"Thư mục nguồn không tồn tại: {source_dir}")

Đã sao chép thành công toàn bộ thư mục từ /content/absa-runs sang /content/drive/MyDrive/absa-runs


## Results

In [22]:
import json
import subprocess
import sys
from pathlib import Path

expected = [
    f"{candidate}-seed-{seed}"
    for candidate in ("PHOBERT-01", "PHOBERT-02")
    for seed in (42, 2026, 3407)
]

ready = []
for run_id in expected:
    run_dir = Path(local_runs) / run_id
    result_path = run_dir / "run_result.json"

    if result_path.is_file():
        result = json.loads(result_path.read_text())
        ok = result.get("status") == "COMPLETED" and result.get("test_evaluated") is False
        if ok:
            ready.append(run_id)
            print(
                run_id, "COMPLETED",
                "Macro-F1 =", result["val_joint_macro_f1"],
                "Micro-F1 =", result["val_joint_micro_f1"],
            )
        else:
            print(run_id, "INVALID RESULT")
    else:
        failed = list(run_dir.glob("run-event-*-failed.json"))
        print(run_id, "FAILED" if failed else "CHƯA CÓ KẾT QUẢ")

print(f"\nPhoBERT hoàn tất: {len(ready)}/6")

selection_path = Path(local_runs) / "family_selection" / "phobert.json"

if len(ready) < 6:
    print("Chưa aggregate được: runner yêu cầu đủ 6 kết quả.")
elif selection_path.exists():
    print("Aggregate đã tồn tại:", selection_path)
    print(selection_path.read_text())
else:
    p = subprocess.run(
        [
            sys.executable, "-u", "-m", "src.training.registered_run", "aggregate",
            "--family", "phobert",
            "--artifact-root", str(local_runs),
        ],
        cwd=repo_dir,
        capture_output=True,
        text=True,
        check=False,
    )
    print("Aggregate exit code:", p.returncode)
    print("STDOUT:\n", p.stdout or "(trống)")
    print("STDERR:\n", p.stderr or "(trống)")

PHOBERT-01-seed-42 COMPLETED Macro-F1 = 0.5229106292974356 Micro-F1 = 0.8636067708333334
PHOBERT-01-seed-2026 CHƯA CÓ KẾT QUẢ
PHOBERT-01-seed-3407 CHƯA CÓ KẾT QUẢ
PHOBERT-02-seed-42 CHƯA CÓ KẾT QUẢ
PHOBERT-02-seed-2026 CHƯA CÓ KẾT QUẢ
PHOBERT-02-seed-3407 CHƯA CÓ KẾT QUẢ

PhoBERT hoàn tất: 1/6
Chưa aggregate được: runner yêu cầu đủ 6 kết quả.
